# Favorita Temporal Fusion Transformer evaluation (Google Colab GPU)

This notebook trains a compact TemporalFusionTransformer on the prepared 116-series Favorita export. It produces two separately labelled evaluations:

- **Primary validation:** 116 item-store series × 28 dates (3,248 observations), rolling one-step-ahead with actual sales through the previous day. This matches the main project validation output and its Naive Weekly and LightGBM evaluation keys.
- **Model-comparison holdout:** 6 series × the same 28 dates (168 observations), recursively forecast from the July 18 origin. This matches the separate model-comparison output's recursive 28-day protocol. This score is reported separately and is the closer comparison to that benchmark.

The model-selection block is June 21–July 4 and is used only for early stopping. The final refit uses target history through July 18. July 19–August 15 remains the held-out evaluation period. Training, checkpoint selection, and normalization never use the final evaluation outcomes.

## Files to upload in Colab

Open `notebooks/tft_colab.ipynb` in Google Colab. Upload these four data/reference files when prompted:

1. `data/processed/tft_favorita_116_series.csv`
2. `notebooks/tft_dataset_manifest.json`
3. `outputs/forecasts/validation.csv`
4. `outputs/forecasts/model_comparison_validation.csv`

The two validation CSVs are reference keys/actuals only; their prediction columns are never model inputs. The notebook verifies file hashes, series/date keys, and actual values before fitting. The 44 MB dataset is suitable for Colab's file upload. For repeat runs, place the same files in Google Drive and replace the upload cell with Drive paths without changing their filenames.

In Colab, select **Runtime → Change runtime type → T4 GPU**, then run the notebook from top to bottom. A GPU check stops the notebook before dataset/model construction if CUDA is unavailable.

No TFT metrics are included in the repository or dashboard by this notebook. The downloadable metrics and predictions are created only after successful GPU training and held-out scoring. The point prediction from QuantileLoss is the median quantile.


In [ ]:
%pip install -q "pytorch-forecasting>=1.4,<2" "lightning>=2.4,<3" "pandas>=2,<3.1" "scikit-learn>=1.3,<2

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
REQUIRED_UPLOADS = {
    "tft_favorita_116_series.csv",
    "tft_dataset_manifest.json",
    "validation.csv",
    "model_comparison_validation.csv",
}
missing_uploads = sorted(REQUIRED_UPLOADS - set(uploaded))
if missing_uploads:
    raise FileNotFoundError(
        "Upload the prepared export, its matching manifest, and both validation reference CSVs. "
        f"Missing: {missing_uploads}"
    )
DATA_PATH = Path("tft_favorita_116_series.csv")
MANIFEST_PATH = Path("tft_dataset_manifest.json")
PRIMARY_VALIDATION_PATH = Path("validation.csv")
COMPARISON_VALIDATION_PATH = Path("model_comparison_validation.csv")
OUTPUT_DIR = Path("/content/tft_colab_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / "tft_execution_report.json"
print("Dataset:", DATA_PATH, f"({DATA_PATH.stat().st_size / 1_000_000:.1f} MB)")
print("Manifest:", MANIFEST_PATH)
print("Output directory:", OUTPUT_DIR)

In [ ]:
import hashlib
import json
import math
import random
import re
import shutil
import traceback
import uuid
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
RUN_ID = uuid.uuid4().hex
REPORT = {
    "run_id": RUN_ID,
    "status": "running",
    "stage": "dependency_and_gpu_check",
    "model": "TemporalFusionTransformer",
    "seed": SEED,
    "training_succeeded": False,
    "heldout_evaluation_succeeded": False,
}

def write_report(status=None, stage=None, error=None, **fields):
    if status is not None:
        REPORT["status"] = status
    if stage is not None:
        REPORT["stage"] = stage
    if error is not None:
        REPORT["error"] = error
    elif status in ("running", "success"):
        REPORT.pop("error", None)
    REPORT.update(fields)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_PATH.write_text(json.dumps(REPORT, indent=2, default=str))

try:
    import torch
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
    from lightning.pytorch.loggers import CSVLogger
    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.data.encoders import NaNLabelEncoder
    from pytorch_forecasting.metrics import QuantileLoss
except Exception as exc:
    write_report(status="failed", stage="dependency_import", error=f"{type(exc).__name__}: {exc}")
    raise

if not torch.cuda.is_available():
    message = "CUDA GPU is unavailable. Select Runtime > Change runtime type > T4 GPU, then rerun all cells."
    write_report(status="failed", stage="gpu_check", error=message,
                 cuda_available=False, device="CPU; training was not attempted")
    raise RuntimeError(message)

torch.set_num_threads(min(4, torch.get_num_threads()))
pl.seed_everything(SEED, workers=True)
GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
REPORT.update({
    "cuda_available": True,
    "gpu_name": GPU_NAME,
    "gpu_compute_capability": list(GPU_CAPABILITY),
    "torch_version": torch.__version__,
    "lightning_version": pl.__version__,
    "pytorch_forecasting_version": __import__("pytorch_forecasting").__version__,
    "device": "cuda:0",
})
write_report(stage="gpu_ready")
print("CUDA active:", torch.cuda.is_available(), "| GPU:", GPU_NAME, "| capability:", GPU_CAPABILITY)
print("PyTorch:", torch.__version__, "Lightning:", pl.__version__,
      "PyTorch Forecasting:", __import__("pytorch_forecasting").__version__)

## Verify the prepared data, manifest, and held-out references

The export contains daily Favorita actual sales, prior-sales lag/rolling features, known calendar/promotion inputs, lagged oil/transaction covariates, static item/store categories, and a training-only RMSSE scale. The target `unit_sales` is an unknown time-varying target and is never a known input.

The first 28 dates per series have structurally undefined lag/rolling features because no 28-day history exists yet. Those 3,248 warm-up rows stay intact in the source export, including their unchanged sales targets, but are excluded as TFT windows. They are not backfilled or imputed. Other numeric inputs are filled only from a deterministic binary default or from earlier values (with a one-day shift for raw oil/transaction values). Calendar categories are reconstructed from each row's date; missing static categories receive the explicit `__MISSING__` category. The cell reports raw missing counts by feature family and then asserts every feature is valid in the retained training/validation rows.


In [ ]:
RESULT_FILES = [
    "tft_predictions.csv", "tft_model_comparison_predictions.csv", "tft_metrics.json",
    "tft_model.ckpt", "tft_training_history.csv", "tft_colab_artifacts.zip",
]
for name in RESULT_FILES:
    path = OUTPUT_DIR / name
    if path.exists():
        path.unlink()

DATA_VALIDATION_SUCCEEDED = False
DATASET_READY = False
TRAINING_SUCCEEDED = False
EVALUATION_SUCCEEDED = False
best_model = None
train_loader = None
selection_loader = None
final_eval_loader = None

try:
    manifest = json.loads(MANIFEST_PATH.read_text())
    file_hashes = {
        "dataset": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
        "primary_validation": hashlib.sha256(PRIMARY_VALIDATION_PATH.read_bytes()).hexdigest(),
        "model_comparison_validation": hashlib.sha256(COMPARISON_VALIDATION_PATH.read_bytes()).hexdigest(),
    }
    expected_hashes = {
        "dataset": manifest["export_sha256"],
        "primary_validation": manifest["primary_validation_reference"]["sha256"],
        "model_comparison_validation": manifest["model_comparison_reference"]["sha256"],
    }
    if file_hashes != expected_hashes:
        raise ValueError(f"Uploaded file hash mismatch. Expected {expected_hashes}; received {file_hashes}")

    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    primary_reference = pd.read_csv(PRIMARY_VALIDATION_PATH, parse_dates=["date"])
    comparison_reference = pd.read_csv(COMPARISON_VALIDATION_PATH, parse_dates=["date"])
    required = set(manifest["export_columns"])
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Export is missing manifest columns: {missing}")
    df["series_id"] = df.series_id.astype(str)
    df["id"] = df.id.astype(str)
    df = df.sort_values(["series_id", "time_idx"]).reset_index(drop=True)
    if manifest.get("manifest_schema_version") != 2:
        raise ValueError("Manifest schema is not the verified version 2 handoff")

    VALIDATION_START = pd.Timestamp(manifest["validation_period"]["start"])
    VALIDATION_END = pd.Timestamp(manifest["validation_period"]["end"])
    SELECTION_START = pd.Timestamp(manifest["model_selection_period"]["start"])
    SELECTION_END = pd.Timestamp(manifest["model_selection_period"]["end"])
    TRAIN_END = VALIDATION_START - pd.Timedelta(days=1)
    EXPECTED_SERIES = int(manifest["series_count"])
    EXPECTED_PRIMARY_ROWS = int(manifest["primary_validation_reference"]["observations"])
    EXPECTED_COMPARISON_ROWS = int(manifest["model_comparison_reference"]["observations"])
    EXPECTED_SELECTION_ROWS = EXPECTED_SERIES * int((SELECTION_END - SELECTION_START).days + 1)
    EXPECTED_PRIMARY_ROWS_FROM_MANIFEST = EXPECTED_SERIES * int(manifest["validation_period"]["days"])
    if EXPECTED_PRIMARY_ROWS != EXPECTED_PRIMARY_ROWS_FROM_MANIFEST:
        raise ValueError("Primary reference row count disagrees with the declared validation scope")

    expected_ids = set(map(str, manifest["series_ids"]))
    actual_ids = set(df.series_id.unique())
    if actual_ids != expected_ids or len(actual_ids) != EXPECTED_SERIES:
        raise ValueError("Export series IDs do not exactly match the manifest")
    if not (df.series_id == df.id).all():
        raise ValueError("series_id and Favorita id do not match")
    if df.duplicated(["series_id", "date"]).any():
        raise ValueError("Duplicate item-store/date keys in export")
    if df.unit_sales.isna().any() or (df.unit_sales < 0).any():
        raise ValueError("Target has missing or negative sales values")

    dates = pd.date_range(df.date.min(), df.date.max(), freq="D")
    expected_per_series = len(dates)
    counts = df.groupby("series_id", observed=True).size()
    if not counts.eq(expected_per_series).all():
        raise ValueError("Series do not have a complete common daily time grid")
    time_date_pairs = df[["time_idx", "date"]].drop_duplicates()
    if time_date_pairs.time_idx.nunique() != time_date_pairs.date.nunique():
        raise ValueError("time_idx does not map one-to-one to calendar dates")
    time_to_date = dict(zip(time_date_pairs.time_idx.astype(int), time_date_pairs.date))

    validation_actuals = df[df.date.between(VALIDATION_START, VALIDATION_END)].copy()
    if len(validation_actuals) != EXPECTED_PRIMARY_ROWS:
        raise ValueError(f"Expected {EXPECTED_PRIMARY_ROWS} primary held-out rows, got {len(validation_actuals)}")
    expected_validation_keys = validation_actuals[["id", "date"]].sort_values(["id", "date"]).reset_index(drop=True)
    primary_keys = primary_reference[["id", "date"]].assign(id=lambda x: x.id.astype(str)).sort_values(["id", "date"]).reset_index(drop=True)
    if not expected_validation_keys.equals(primary_keys):
        raise ValueError("Primary validation reference keys differ from the TFT export")
    primary_actual_check = primary_reference.assign(id=lambda x: x.id.astype(str)).merge(
        validation_actuals[["id", "date", "unit_sales"]], on=["id", "date"], how="left", validate="one_to_one")
    if primary_actual_check.unit_sales.isna().any() or not np.array_equal(
            primary_actual_check.actual.to_numpy(dtype=float), primary_actual_check.unit_sales.to_numpy(dtype=float)):
        raise ValueError("Primary validation actuals do not match the prepared Favorita export")

    comparison_ids = set(map(str, manifest["model_comparison_reference"]["series_ids"]))
    comparison_reference["id"] = comparison_reference.id.astype(str)
    if set(comparison_reference.id.unique()) != comparison_ids:
        raise ValueError("Model-comparison series IDs differ from the manifest")
    if len(comparison_reference) != EXPECTED_COMPARISON_ROWS:
        raise ValueError("Model-comparison reference row count differs from the manifest")
    comparison_actual_check = comparison_reference.merge(
        df[["id", "date", "unit_sales"]], on=["id", "date"], how="left", validate="one_to_one")
    if comparison_actual_check.unit_sales.isna().any() or not np.array_equal(
            comparison_actual_check.actual.to_numpy(dtype=float), comparison_actual_check.unit_sales.to_numpy(dtype=float)):
        raise ValueError("Model-comparison actuals do not match the prepared Favorita export")
    expected_comparison_keys = validation_actuals[validation_actuals.id.isin(comparison_ids)][["id", "date"]].sort_values(["id", "date"]).reset_index(drop=True)
    comparison_keys = comparison_reference[["id", "date"]].sort_values(["id", "date"]).reset_index(drop=True)
    if not expected_comparison_keys.equals(comparison_keys):
        raise ValueError("Recursive model-comparison holdout keys differ from the TFT export")

    target_before_preprocessing = df[["id", "date", "unit_sales"]].copy()
    all_known_reals = list(manifest["tft_known_real_covariates"])
    known_categoricals = list(manifest["tft_known_categorical_covariates"])
    static_categoricals = list(manifest["tft_static_categorical_covariates"])
    required_features = all_known_reals + known_categoricals + static_categoricals
    missing_before = {column: int(df[column].isna().sum()) for column in required_features}
    df["time_idx"] = pd.to_numeric(df.time_idx, errors="coerce")
    if df.time_idx.isna().any():
        origin = pd.Timestamp(manifest["time_index_origin"])
        derived_time_idx = (df.date - origin).dt.days
        df["time_idx"] = df.time_idx.fillna(derived_time_idx)
    df["time_idx"] = df.time_idx.astype("int64")

    # Binary calendar/promotion flags: missing means no event/promotion.
    binary_real_fills = {}
    for column in ("onpromotion", "is_holiday"):
        df[column] = pd.to_numeric(df[column], errors="coerce")
        binary_real_fills[column] = int(df[column].isna().sum())
        df[column] = df[column].fillna(0.0)

    # Lagged oil/transactions values are repaired only from information available
    # on earlier dates; never use bfill or a statistic computed from future rows.
    causal_numeric_fills = {}
    for column, raw_column in (("oil_price_lag_1", "oil_price"),
                               ("transactions_lag_1", "transactions")):
        df[column] = pd.to_numeric(df[column], errors="coerce")
        before_count = int(df[column].isna().sum())
        prior_raw = pd.to_numeric(df[raw_column], errors="coerce").groupby(
            df.series_id, sort=False, observed=True).shift(1)
        df[column] = df[column].fillna(prior_raw)
        df[column] = df.groupby("series_id", sort=False, observed=True)[column].ffill()
        causal_numeric_fills[column] = before_count - int(df[column].isna().sum())

    # Calendar categoricals are deterministic functions of the same row's date.
    calendar_values = {
        "dow_cat": df.date.dt.dayofweek.astype("string"),
        "month_cat": df.date.dt.month.astype("string"),
        "day_cat": df.date.dt.day.astype("string"),
        "weekofyear_cat": df.date.dt.isocalendar().week.astype("string"),
    }
    categorical_fills = {}
    for column in known_categoricals:
        values = df[column].astype("string")
        before_count = int(values.isna().sum())
        values = values.fillna(calendar_values[column])
        values = values.fillna("__MISSING__").astype(str)
        df[column] = values
        categorical_fills[column] = before_count
    for column in static_categoricals:
        values = df[column].astype("string")
        before_count = int(values.isna().sum())
        df[column] = values.fillna("__MISSING__").astype(str)
        categorical_fills[column] = before_count

    # Lag/rolling nulls are valid only in the leading 28-day feature warm-up.
    # Reject any internal gaps rather than silently imputing them.
    warmup_lengths = {"lag_1": 1, "lag_7": 7, "lag_14": 14, "lag_28": 28,
                      "mean_7": 7, "mean_28": 28, "std_28": 28}
    structural_missing = {column: int(df[column].isna().sum()) for column in warmup_lengths}
    for series_id, group in df.groupby("series_id", sort=False, observed=True):
        for column, leading_count in warmup_lengths.items():
            missing_positions = np.flatnonzero(group[column].isna().to_numpy())
            if not np.array_equal(missing_positions, np.arange(leading_count)):
                raise ValueError(
                    f"{column} has missing values outside its leading {leading_count}-row warm-up for {series_id}; "
                    "refusing to impute a broken lag/rolling feature."
                )
    warmup_mask = df.groupby("series_id", sort=False, observed=True).cumcount() < 28
    model_df = df.loc[~warmup_mask].copy()
    known_reals = [column for column in all_known_reals if column != "time_idx"]
    numeric_known = ["time_idx"] + known_reals
    if model_df[required_features].isna().any().any():
        bad = {column: int(model_df[column].isna().sum()) for column in required_features
               if model_df[column].isna().any()}
        raise ValueError(f"Required TFT features remain missing after leakage-safe preprocessing: {bad}")
    nonfinite = [column for column in numeric_known
                 if not np.isfinite(pd.to_numeric(model_df[column], errors="coerce").to_numpy(dtype=float)).all()]
    if nonfinite:
        raise ValueError(f"Required numeric TFT features contain non-finite values: {nonfinite}")
    if model_df.unit_sales.isna().any():
        raise ValueError("Target values must remain present after feature warm-up exclusion")
    if not target_before_preprocessing.equals(df[["id", "date", "unit_sales"]]):
        raise AssertionError("Feature preprocessing changed or reordered the source target values")
    missing_feature_report = {
        "before_preprocessing": missing_before,
        "structural_lag_rolling_warmup": structural_missing,
        "binary_numeric_filled_with_zero": binary_real_fills,
        "causal_lagged_covariate_values_repaired_from_past": causal_numeric_fills,
        "calendar_categorical_values_reconstructed_from_date": {
            column: categorical_fills[column] for column in known_categoricals},
        "static_categorical_values_filled_with_missing_token": {
            column: categorical_fills[column] for column in static_categoricals},
        "warmup_rows_excluded_from_windows": int(warmup_mask.sum()),
        "retained_rows_with_missing_required_features": 0,
        "target_values_modified": False,
    }
    print("Raw missing required features by family:")
    print("Known real:", {c:n for c,n in missing_before.items() if c in all_known_reals and n})
    print("Known categorical:", {c:n for c,n in missing_before.items() if c in known_categoricals and n})
    print("Static categorical:", {c:n for c,n in missing_before.items() if c in static_categoricals and n})
    print("Warm-up feature nulls:", structural_missing)
    print("Rows retained with all required features valid:", len(model_df))
    if not model_df[required_features].notna().all().all():
        raise AssertionError("Pre-training feature-validity assertion failed")
    REPORT["feature_missingness_and_preprocessing"] = missing_feature_report

    # Confirm the one-step lag feature uses the prior date's actual sales.
    previous = df[["id", "date", "unit_sales"]].copy()
    previous["date"] += pd.Timedelta(days=1)
    lag_check = validation_actuals[["id", "date", "lag_1"]].merge(
        previous, on=["id", "date"], how="left", validate="one_to_one")
    if lag_check.unit_sales.isna().any() or not np.array_equal(
            lag_check.lag_1.to_numpy(dtype=float), lag_check.unit_sales.to_numpy(dtype=float)):
        raise ValueError("Validation lag_1 is not exactly prior-day sales")

    # Verify the RMSSE denominator is computed strictly before the final test period.
    scale_check = df[df.date < VALIDATION_START].sort_values(["series_id", "date"]).groupby("series_id", observed=True).unit_sales.apply(
        lambda values: float(np.mean(np.diff(values.to_numpy(dtype=float)) ** 2)))
    exported_scale = df.groupby("series_id", observed=True).rmsse_scale.first().astype(float)
    if not np.allclose(scale_check.sort_index(), exported_scale.sort_index(), rtol=1e-8, atol=1e-10):
        raise ValueError("Exported RMSSE scales do not match pre-test training history")
    if (exported_scale <= 0).any():
        raise ValueError("RMSSE scale must be positive for every series")

    model_df = df.dropna(subset=known_reals + ["time_idx"]).copy()
    training_rows = model_df[model_df.date < SELECTION_START].copy()
    selection_rows = model_df[model_df.date <= SELECTION_END].copy()
    refit_rows = model_df[model_df.date < VALIDATION_START].copy()
    if training_rows.date.max() != SELECTION_START - pd.Timedelta(days=1):
        raise ValueError("Early-stopping model training does not end immediately before the selection period")
    if refit_rows.date.max() != TRAIN_END:
        raise ValueError("Final model refit does not end immediately before the held-out period")

    REPORT.update({
        "dataset_file": DATA_PATH.name,
        "manifest_file": MANIFEST_PATH.name,
        "reference_files": [PRIMARY_VALIDATION_PATH.name, COMPARISON_VALIDATION_PATH.name],
        "sha256": file_hashes,
        "series_count": EXPECTED_SERIES,
        "training_start": str(training_rows.date.min().date()),
        "early_stopping_training_end": str(training_rows.date.max().date()),
        "model_selection_start": str(SELECTION_START.date()),
        "model_selection_end": str(SELECTION_END.date()),
        "refit_training_end": str(refit_rows.date.max().date()),
        "validation_start": str(VALIDATION_START.date()),
        "validation_end": str(VALIDATION_END.date()),
        "primary_validation_observations": len(validation_actuals),
        "model_comparison_observations": len(comparison_reference),
        "model_comparison_series_count": len(comparison_ids),
        "data_validation_succeeded": True,
        "leakage_checks": {
            "manifest_hashes_match": True,
            "export_series_ids_match_manifest": True,
            "primary_validation_keys_and_actuals_match": True,
            "recursive_comparison_keys_and_actuals_match": True,
            "lag_1_matches_prior_actual": True,
            "target_excluded_from_known_covariates": True,
            "RMSSE_scale_uses_pre_test_history_only": True,
        },
    })
    DATA_VALIDATION_SUCCEEDED = True
    write_report(stage="data_validation_complete", data_validation_succeeded=True)
    print(f"Verified {EXPECTED_SERIES} series; {len(validation_actuals):,} primary one-step rows; "
          f"{len(comparison_reference):,} recursive comparison rows.")
    print(f"Selection for early stopping: {SELECTION_START.date()} through {SELECTION_END.date()}; "
          f"final refit through {TRAIN_END.date()}; untouched test: {VALIDATION_START.date()} through {VALIDATION_END.date()}.")
except Exception as exc:
    write_report(status="failed", stage="data_validation", error=f"{type(exc).__name__}: {exc}",
                 data_validation_succeeded=False)
    raise

## Build chronological TimeSeriesDataSet objects

The validated `model_df` is the only input to dataset construction. Its 3,248 first-28-day warm-up rows are excluded because the 28-day lag/rolling inputs are undefined there; their source rows and target values are preserved in `df`. Missing binary flags use 0, lagged oil/transaction values use only prior observations, calendar categories derive from date, and missing static categories use `__MISSING__` with encoders configured to accept it. The notebook fails if any required feature remains null or non-finite.

Early stopping observes only the project model-selection window (June 21–July 4). After selecting an epoch count there, the final TFT is reinitialized and refit on all pre-test data through July 18. The final 28-day period is used only for predictions and scoring. Each decoder has a one-day horizon. The separate six-series benchmark is generated recursively after the July 18 origin; lag and rolling-sales features in that recursive path are recomputed from TFT predictions rather than future actual sales.


In [ ]:
DATASET_READY = False
TRAINING_SUCCEEDED = False
EVALUATION_SUCCEEDED = False
best_model = None
train_loader = None
selection_loader = None
final_eval_loader = None
try:
    if not DATA_VALIDATION_SUCCEEDED:
        raise RuntimeError("Data validation failed; rerun the upload, GPU-check, and data-validation cells.")
    required_features = ["time_idx"] + known_reals + known_categoricals + static_categoricals
    invalid = {column: int(model_df[column].isna().sum()) for column in required_features
               if model_df[column].isna().any()}
    if invalid:
        raise ValueError(f"Pre-training feature validation failed: {invalid}")
    if model_df[required_features].isna().any().any():
        raise AssertionError("All required real and categorical features must be valid before TimeSeriesDataSet creation")
    if model_df.unit_sales.isna().any():
        raise AssertionError("Target values were unexpectedly altered or removed")
    selection_train_rows = model_df[model_df.date < SELECTION_START].copy()
    selection_window_rows = model_df[model_df.date <= SELECTION_END].copy()
    final_train_rows = model_df[model_df.date < VALIDATION_START].copy()
    selection_start_idx = int(model_df.loc[model_df.date.eq(SELECTION_START), "time_idx"].iloc[0])
    evaluation_start_idx = int(model_df.loc[model_df.date.eq(VALIDATION_START), "time_idx"].iloc[0])
    ENCODER_DAYS = 56
    BATCH_SIZE = 32
    TRAIN_BATCH_FRACTION = 0.15

    def build_training_dataset(rows):
        return TimeSeriesDataSet(
            rows,
            time_idx="time_idx",
            target="unit_sales",
            group_ids=["series_id"],
            min_encoder_length=ENCODER_DAYS,
            max_encoder_length=ENCODER_DAYS,
            min_prediction_length=1,
            max_prediction_length=1,
            static_categoricals=static_categoricals,
            time_varying_known_categoricals=known_categoricals,
            time_varying_known_reals=["time_idx"] + known_reals,
            time_varying_unknown_reals=["unit_sales"],
            target_normalizer=GroupNormalizer(groups=["series_id"]),
            categorical_encoders={
                column: NaNLabelEncoder(add_nan=True)
                for column in static_categoricals + known_categoricals
            },
            add_relative_time_idx=True,
            add_target_scales=True,
            add_encoder_length=True,
            allow_missing_timesteps=False,
            randomize_length=False,
        )

    selection_train_dataset = build_training_dataset(selection_train_rows)
    selection_dataset = TimeSeriesDataSet.from_dataset(
        selection_train_dataset,
        selection_window_rows,
        min_prediction_idx=selection_start_idx,
        predict=False,
        stop_randomization=True,
    )
    expected_selection_windows = EXPECTED_SERIES * int((SELECTION_END - SELECTION_START).days + 1)
    if len(selection_dataset) != expected_selection_windows:
        raise ValueError(f"Expected {expected_selection_windows} early-stopping windows; got {len(selection_dataset)}")
    selection_train_loader = selection_train_dataset.to_dataloader(
        train=True, batch_size=BATCH_SIZE, num_workers=0)
    selection_loader = selection_dataset.to_dataloader(
        train=False, batch_size=BATCH_SIZE, num_workers=0)

    final_train_dataset = build_training_dataset(final_train_rows)
    train_loader = final_train_dataset.to_dataloader(
        train=True, batch_size=BATCH_SIZE, num_workers=0)
    final_evaluation_dataset = TimeSeriesDataSet.from_dataset(
        final_train_dataset,
        model_df,
        min_prediction_idx=evaluation_start_idx,
        predict=False,
        stop_randomization=True,
    )
    if len(final_evaluation_dataset) != EXPECTED_PRIMARY_ROWS:
        raise ValueError(f"Expected {EXPECTED_PRIMARY_ROWS} final rolling windows; got {len(final_evaluation_dataset)}")
    final_eval_loader = final_evaluation_dataset.to_dataloader(
        train=False, batch_size=BATCH_SIZE, num_workers=0)
    DATASET_READY = True
    REPORT.update({
        "encoder_length_days": ENCODER_DAYS,
        "forecast_horizon_days": 1,
        "batch_size": BATCH_SIZE,
        "train_batch_fraction_per_epoch": TRAIN_BATCH_FRACTION,
        "early_stopping_windows": len(selection_dataset),
        "final_refit_training_windows": len(final_train_dataset),
        "primary_heldout_windows": len(final_evaluation_dataset),
        "known_real_covariates": ["time_idx"] + known_reals,
        "known_categorical_covariates": known_categoricals,
        "static_categorical_covariates": static_categoricals,
    })
    write_report(stage="dataset_construction_complete", dataset_ready=True)
    print("Selection windows:", len(selection_dataset), "Final training windows:", len(final_train_dataset),
          "Final held-out one-step windows:", len(final_evaluation_dataset))
except Exception as exc:
    DATASET_READY = False
    write_report(status="failed", stage="dataset_construction", error=f"{type(exc).__name__}: {exc}",
                 dataset_ready=False)
    raise

## Train and refit the compact GPU TFT

The selection run uses EarlyStopping and validation loss on June 21–July 4. The final pre-test refit is a separate cell so it can be retried without repeating selection. Its optimizer/scheduler compatibility handling is documented in the refit cell.


In [ ]:
SELECTION_SUCCEEDED = False
TRAINING_SUCCEEDED = False
EVALUATION_SUCCEEDED = False
best_model = None
for name in ("tft_predictions.csv", "tft_model_comparison_predictions.csv", "tft_metrics.json",
             "tft_model.ckpt", "tft_training_history.csv", "tft_colab_artifacts.zip"):
    path = OUTPUT_DIR / name
    if path.exists():
        path.unlink()
REPORT.pop("metrics_file", None)
REPORT.pop("predictions_file", None)
write_report(status="running", stage="selection_training", selection_succeeded=False,
             training_succeeded=False, heldout_evaluation_succeeded=False)
try:
    if not DATASET_READY or selection_loader is None or selection_train_loader is None:
        raise RuntimeError("Training datasets are not ready; run data validation and dataset construction first.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA became unavailable before training; select a Colab T4 GPU and rerun.")
    GPU_NAME = torch.cuda.get_device_name(0)
    ATTEMPT_ID = uuid.uuid4().hex
    checkpoint_root = OUTPUT_DIR / "selection_checkpoints" / ATTEMPT_ID
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    selection_logger = CSVLogger(save_dir=str(OUTPUT_DIR), name=f"selection_logs_{ATTEMPT_ID}")
    selection_checkpoint = ModelCheckpoint(
        dirpath=str(checkpoint_root), filename="tft-selection-{epoch:02d}",
        monitor="val_loss", mode="min", save_top_k=1,
    )
    early_stopping = EarlyStopping(
        monitor="val_loss", min_delta=1e-4, patience=3, mode="min", verbose=True)

    def create_tft(dataset, use_plateau_scheduler=True):
        return TemporalFusionTransformer.from_dataset(
            dataset,
            learning_rate=3e-3,
            hidden_size=8,
            attention_head_size=2,
            dropout=0.1,
            hidden_continuous_size=8,
            loss=QuantileLoss(),
            log_interval=100,
            reduce_on_plateau_patience=2 if use_plateau_scheduler else None,
        )

    selection_model = create_tft(selection_train_dataset)
    selection_trainer = pl.Trainer(
        max_epochs=15,
        accelerator="gpu",
        devices=1,
        callbacks=[early_stopping, selection_checkpoint],
        logger=selection_logger,
        gradient_clip_val=0.1,
        enable_model_summary=False,
        log_every_n_steps=100,
        deterministic=True,
        enable_checkpointing=True,
        limit_train_batches=TRAIN_BATCH_FRACTION,
    )
    selection_trainer.fit(
        selection_model,
        train_dataloaders=selection_train_loader,
        val_dataloaders=selection_loader,
    )
    if selection_trainer.strategy.root_device.type != "cuda":
        raise RuntimeError(f"Expected CUDA training, got {selection_trainer.strategy.root_device}")
    selection_history_path = Path(selection_logger.log_dir) / "metrics.csv"
    if not selection_history_path.is_file() or not selection_checkpoint.best_model_path:
        raise RuntimeError("Selection training did not create a history file and monitored checkpoint")
    selection_history = pd.read_csv(selection_history_path)
    if "val_loss" not in selection_history or not selection_history.val_loss.notna().any():
        raise RuntimeError("Selection history contains no recorded validation loss")
    best_selection_row = selection_history.loc[selection_history.val_loss.idxmin()]
    BEST_EPOCHS = int(best_selection_row.epoch) + 1
    if BEST_EPOCHS < 1 or BEST_EPOCHS > 15:
        raise ValueError(f"Invalid early-stopped epoch count: {BEST_EPOCHS}")

    SELECTION_SUCCEEDED = True
    REPORT.update({
        "selection_succeeded": True,
        "selection_checkpoint": selection_checkpoint.best_model_path,
        "selection_best_epoch": BEST_EPOCHS,
        "selection_best_validation_loss": float(best_selection_row.val_loss),
        "selection_epochs_completed": int(selection_trainer.current_epoch + 1),
    })
    write_report(status="running", stage="selection_complete", training_succeeded=False,
                 selection_succeeded=True)
    print("Selection complete on CUDA:", GPU_NAME)
    print("Selected refit epochs:", BEST_EPOCHS, "| best validation loss:", float(best_selection_row.val_loss))
except Exception as exc:
    SELECTION_SUCCEEDED = False
    TRAINING_SUCCEEDED = False
    write_report(status="failed", stage="selection_training", selection_succeeded=False,
                 training_succeeded=False, error=f"{type(exc).__name__}: {exc}")
    raise


## Final pre-test refit (scheduler-free)

PyTorch Forecasting 1.8.0 returns an empty `lr_scheduler` mapping when `reduce_on_plateau_patience=None`. Lightning 2.6.6 rejects that mapping with a scheduler-dict misconfiguration. The refit model below subclasses TFT only to remove that empty scheduler entry, preserving PyTorch Forecasting's configured optimizer. It returns a valid optimizer-only configuration. The early-stopping selection run above keeps its valid ReduceLROnPlateau scheduler. Architecture, learning rate, epoch count, data split, and evaluation protocol are unchanged.

If the selection stage already completed in the current Colab runtime, this cell can be rerun by itself after replacing/updating the notebook; it verifies the retained selection checkpoint and history. If starting a fresh runtime, run the selection cell first.


In [ ]:
TRAINING_SUCCEEDED = False
EVALUATION_SUCCEEDED = False
best_model = None
try:
    prior_selection_state = (
        "BEST_EPOCHS" in globals()
        and "selection_history" in globals()
        and "selection_checkpoint" in globals()
        and bool(selection_checkpoint.best_model_path)
        and Path(selection_checkpoint.best_model_path).is_file()
    )
    if not (globals().get("SELECTION_SUCCEEDED", False) or prior_selection_state):
        raise RuntimeError("No successful selection run is available; run the early-stopping selection cell first.")
    if not DATASET_READY or train_loader is None or final_train_dataset is None:
        raise RuntimeError("Final pre-test training data is not ready; run the dataset-construction cell first.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA became unavailable before refit; select a Colab T4 GPU and rerun.")
    if not 1 <= int(BEST_EPOCHS) <= 15:
        raise ValueError(f"Invalid selection-derived refit epoch count: {BEST_EPOCHS}")

    # Release model/trainer objects from a prior failed refit before retrying.
    for variable_name in ("selection_model", "selection_trainer", "final_model", "refit_trainer"):
        if variable_name in globals():
            del globals()[variable_name]
    torch.cuda.empty_cache()

    class TFTWithoutLRScheduler(TemporalFusionTransformer):
        def configure_optimizers(self):
            # PF 1.8.0 creates lr_scheduler={} for patience=None; strip that key
            # and preserve the exact optimizer returned by the PF base class.
            config = super().configure_optimizers()
            if isinstance(config, dict):
                optimizer = config.get("optimizer")
                if not isinstance(optimizer, torch.optim.Optimizer):
                    raise TypeError("PyTorch Forecasting did not return a valid optimizer")
                config.pop("lr_scheduler", None)
                return config
            if isinstance(config, torch.optim.Optimizer):
                return config
            raise TypeError(f"Unexpected optimizer configuration type: {type(config).__name__}")

    def create_tft(dataset, use_plateau_scheduler=True):
        model_class = TemporalFusionTransformer if use_plateau_scheduler else TFTWithoutLRScheduler
        return model_class.from_dataset(
            dataset,
            learning_rate=3e-3,
            hidden_size=8,
            attention_head_size=2,
            dropout=0.1,
            hidden_continuous_size=8,
            loss=QuantileLoss(),
            log_interval=100,
            reduce_on_plateau_patience=2 if use_plateau_scheduler else None,
        )

    final_model = create_tft(final_train_dataset, use_plateau_scheduler=False)
    optimizer_config = final_model.configure_optimizers()
    if isinstance(optimizer_config, dict):
        if not isinstance(optimizer_config.get("optimizer"), torch.optim.Optimizer):
            raise TypeError("Scheduler-free refit returned no valid optimizer")
        if optimizer_config.get("lr_scheduler"):
            raise TypeError("Scheduler-free refit unexpectedly returned a learning-rate scheduler")
        refit_optimizer_name = type(optimizer_config["optimizer"]).__name__
    elif isinstance(optimizer_config, torch.optim.Optimizer):
        refit_optimizer_name = type(optimizer_config).__name__
    else:
        raise TypeError(f"Unexpected scheduler-free optimizer config: {type(optimizer_config).__name__}")

    for name in ("tft_predictions.csv", "tft_model_comparison_predictions.csv", "tft_metrics.json",
                 "tft_model.ckpt", "tft_training_history.csv", "tft_colab_artifacts.zip"):
        path = OUTPUT_DIR / name
        if path.exists():
            path.unlink()
    write_report(status="running", stage="final_refit", training_succeeded=False,
                 heldout_evaluation_succeeded=False,
                 refit_optimizer=refit_optimizer_name, refit_scheduler="disabled cleanly (optimizer-only)")
    REFIT_ID = uuid.uuid4().hex
    refit_logger = CSVLogger(save_dir=str(OUTPUT_DIR), name=f"refit_logs_{REFIT_ID}")
    refit_trainer = pl.Trainer(
        max_epochs=BEST_EPOCHS,
        accelerator="gpu",
        devices=1,
        logger=refit_logger,
        gradient_clip_val=0.1,
        enable_model_summary=False,
        log_every_n_steps=100,
        deterministic=True,
        enable_checkpointing=False,
        limit_train_batches=TRAIN_BATCH_FRACTION,
    )
    refit_trainer.fit(final_model, train_dataloaders=train_loader)
    if refit_trainer.strategy.root_device.type != "cuda":
        raise RuntimeError(f"Expected CUDA refit, got {refit_trainer.strategy.root_device}")

    CHECKPOINT_PATH = OUTPUT_DIR / "tft_model.ckpt"
    refit_trainer.save_checkpoint(str(CHECKPOINT_PATH))
    if not CHECKPOINT_PATH.is_file() or CHECKPOINT_PATH.stat().st_size == 0:
        raise RuntimeError("Final CUDA refit checkpoint was not saved")
    best_model = final_model
    best_model.eval()

    refit_history_path = Path(refit_logger.log_dir) / "metrics.csv"
    if not refit_history_path.is_file():
        raise RuntimeError("Final refit training history was not created")
    refit_history = pd.read_csv(refit_history_path)
    selection_history["phase"] = "early_stopping_selection"
    refit_history["phase"] = "final_pre_test_refit"
    history = pd.concat([selection_history, refit_history], ignore_index=True, sort=False)
    HISTORY_PATH = OUTPUT_DIR / "tft_training_history.csv"
    history.to_csv(HISTORY_PATH, index=False)

    TRAINING_SUCCEEDED = True
    REPORT.update({
        "training_succeeded": True,
        "training_run_id": REFIT_ID,
        "gpu_name": torch.cuda.get_device_name(0),
        "architecture": {"hidden_size": 8, "attention_head_size": 2,
                          "hidden_continuous_size": 8, "dropout": 0.1},
        "loss": "QuantileLoss",
        "batch_size": BATCH_SIZE,
        "train_batch_fraction_per_epoch": TRAIN_BATCH_FRACTION,
        "max_epochs": 15,
        "early_stopping_patience": 3,
        "selection_best_epoch": BEST_EPOCHS,
        "selection_best_validation_loss": float(best_selection_row.val_loss),
        "refit_epochs": BEST_EPOCHS,
        "refit_optimizer": refit_optimizer_name,
        "refit_scheduler": "disabled cleanly (optimizer-only)",
        "checkpoint": str(CHECKPOINT_PATH),
        "training_history": str(HISTORY_PATH),
        "selection_checkpoint_used_for_final_scoring": False,
    })
    write_report(status="running", stage="final_refit_complete", training_succeeded=True)
    print("CUDA refit complete on:", torch.cuda.get_device_name(0))
    print("Optimizer:", refit_optimizer_name, "| scheduler: disabled cleanly")
    print("Refit epochs:", BEST_EPOCHS, "| checkpoint:", CHECKPOINT_PATH)
except Exception as exc:
    TRAINING_SUCCEEDED = False
    best_model = None
    for name in ("tft_predictions.csv", "tft_model_comparison_predictions.csv", "tft_metrics.json",
                 "tft_model.ckpt", "tft_training_history.csv", "tft_colab_artifacts.zip"):
        path = OUTPUT_DIR / name
        if path.exists():
            path.unlink()
    write_report(status="failed", stage="final_refit", training_succeeded=False,
                 heldout_evaluation_succeeded=False, error=f"{type(exc).__name__}: {exc}")
    raise


## Predict and evaluate genuine held-out targets

The primary 116-series score uses the same dates and keys as `validation.csv`, with actual prior-day history for each rolling one-step prediction. The separate 6-series score uses a recursive 28-day rollout from July 18, matching `model_comparison_validation.csv`; future target-derived lag/rolling inputs are recomputed from TFT predictions. Actuals and comparison predictions are joined only after inference for evaluation.


In [ ]:
EVALUATION_SUCCEEDED = False
if not TRAINING_SUCCEEDED or best_model is None or not DATASET_READY:
    message = "Evaluation skipped: this run has no successfully GPU-trained final refit."
    write_report(status="failed", stage="prediction_gate", error=message,
                 heldout_evaluation_succeeded=False)
    raise RuntimeError(message)
try:
    write_report(status="running", stage="heldout_prediction_and_scoring",
                 heldout_evaluation_succeeded=False)

    # Main comparison: one-step predictions for all 116 series and 28 dates.
    raw, prediction_index = best_model.predict(
        final_eval_loader,
        mode="prediction",
        return_index=True,
        trainer_kwargs={"accelerator": "gpu", "devices": 1, "logger": False, "enable_progress_bar": False},
    )
    pred_array = raw.detach().cpu().numpy() if torch.is_tensor(raw) else np.asarray(raw)
    pred_array = pred_array.reshape(len(prediction_index), -1)
    if pred_array.shape[1] != 1:
        raise ValueError(f"Expected one quantile median prediction per decoder; got shape {pred_array.shape}")
    if "series_id" not in prediction_index or "time_idx" not in prediction_index:
        raise ValueError(f"Unexpected TFT prediction index columns: {prediction_index.columns.tolist()}")
    prediction_index = prediction_index.copy()
    prediction_index["series_id"] = prediction_index.series_id.astype(str)
    prediction_index["time_idx"] = prediction_index.time_idx.astype(int)
    prediction_index["prediction"] = np.maximum(pred_array[:, 0], 0.0)
    prediction_index["date"] = prediction_index.time_idx.map(time_to_date)
    primary_predictions = prediction_index[["series_id", "date", "prediction"]].rename(columns={"series_id":"id"})
    primary_scored = primary_predictions.merge(
        validation_actuals[["id", "date", "unit_sales", "store_nbr", "item_nbr", "rmsse_scale"]],
        on=["id", "date"], how="left", validate="one_to_one")
    primary_scored = primary_scored.rename(columns={"unit_sales":"actual"})
    if len(primary_scored) != EXPECTED_PRIMARY_ROWS or primary_scored.actual.isna().any():
        raise ValueError("Primary TFT predictions are incomplete or missing held-out actuals")
    if not primary_scored[["id", "date"]].sort_values(["id", "date"]).reset_index(drop=True).equals(
            expected_validation_keys):
        raise ValueError("Primary TFT prediction keys differ from the exact 116-series held-out keys")

    # Recursive protocol for the separate 6-series model-comparison holdout.
    comparison_ids_ordered = sorted(comparison_ids)
    origin = VALIDATION_START - pd.Timedelta(days=1)
    comparison_history = {
        sid: model_df[(model_df.id.eq(sid)) & (model_df.date.le(origin))].sort_values("date").copy()
        for sid in comparison_ids_ordered
    }
    if any(len(part) < ENCODER_DAYS for part in comparison_history.values()):
        raise ValueError("Insufficient encoder history for recursive model-comparison forecast")
    forecast_dates = pd.date_range(VALIDATION_START, VALIDATION_END, freq="D")
    future_covariates = df[df.id.isin(comparison_ids) & df.date.between(VALIDATION_START, VALIDATION_END)].copy()
    future_covariates["id"] = future_covariates.id.astype(str)
    future_covariates = future_covariates.set_index(["id", "date"]).sort_index()
    recursive_rows = []
    static_columns = static_categoricals
    calendar_columns = known_categoricals
    frozen_covariates = {}
    for sid in comparison_ids_ordered:
        origin_row = comparison_history[sid].iloc[-1]
        frozen_covariates[sid] = {
            "oil_price_lag_1": float(origin_row.oil_price),
            "transactions_lag_1": float(origin_row.transactions),
        }

    for horizon_step, day in enumerate(forecast_dates, start=1):
        decoder_rows = []
        for sid in comparison_ids_ordered:
            actual_known_row = future_covariates.loc[(sid, day)]
            if isinstance(actual_known_row, pd.DataFrame):
                raise ValueError(f"Duplicate future known-covariate row for {sid} on {day.date()}")
            state = comparison_history[sid]
            history_targets = state.unit_sales.to_numpy(dtype=float)
            if len(history_targets) < 28:
                raise ValueError(f"Insufficient target history for lag features: {sid}")
            future_row = {column: state.iloc[-1][column] for column in static_columns}
            future_row.update({column: str(actual_known_row[column]) for column in calendar_columns})
            future_row.update({
                "series_id": sid,
                "id": sid,
                "date": day,
                "time_idx": int(actual_known_row.time_idx),
                "unit_sales": 0.0,
                "onpromotion": float(actual_known_row.onpromotion),
                "is_holiday": float(actual_known_row.is_holiday),
                **frozen_covariates[sid],
                "lag_1": float(history_targets[-1]),
                "lag_7": float(history_targets[-7]),
                "lag_14": float(history_targets[-14]),
                "lag_28": float(history_targets[-28]),
                "mean_7": float(np.mean(history_targets[-7:])),
                "mean_28": float(np.mean(history_targets[-28:])),
                "std_28": float(np.std(history_targets[-28:], ddof=1)),
            })
            decoder_rows.append(future_row)
        encoder_rows = pd.concat([comparison_history[sid].tail(ENCODER_DAYS) for sid in comparison_ids_ordered],
                                 ignore_index=True)
        recursive_input = pd.concat([encoder_rows, pd.DataFrame(decoder_rows)], ignore_index=True, sort=False)
        recursive_input = recursive_input.sort_values(["series_id", "time_idx"]).reset_index(drop=True)
        recursive_dataset = TimeSeriesDataSet.from_dataset(
            final_train_dataset, recursive_input, predict=True, stop_randomization=True)
        recursive_loader = recursive_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
        raw_day, day_index = best_model.predict(
            recursive_loader, mode="prediction", return_index=True,
            trainer_kwargs={"accelerator":"gpu", "devices":1, "logger":False, "enable_progress_bar":False})
        day_values = raw_day.detach().cpu().numpy() if torch.is_tensor(raw_day) else np.asarray(raw_day)
        day_values = day_values.reshape(len(day_index), -1)
        if day_values.shape[1] != 1:
            raise ValueError(f"Recursive day {day.date()} returned unexpected prediction shape {day_values.shape}")
        day_index = day_index.copy()
        day_index["series_id"] = day_index.series_id.astype(str)
        pred_map = dict(zip(day_index.series_id, np.maximum(day_values[:, 0], 0.0)))
        if set(pred_map) != comparison_ids:
            raise ValueError(f"Recursive day {day.date()} prediction IDs do not match comparison sample")
        for sid in comparison_ids_ordered:
            prediction = float(pred_map[sid])
            new_row = next(row for row in decoder_rows if row["series_id"] == sid)
            new_row["unit_sales"] = prediction
            comparison_history[sid] = pd.concat(
                [comparison_history[sid], pd.DataFrame([new_row])], ignore_index=True, sort=False)
            reference_row = comparison_reference[
                comparison_reference.id.eq(sid) & comparison_reference.date.eq(day)]
            if len(reference_row) != 1:
                raise ValueError(f"Missing unique model-comparison reference for {sid} on {day.date()}")
            recursive_rows.append({
                "date": day, "id": sid, "prediction": prediction,
                "actual": float(reference_row.iloc[0].actual),
                "forecast_lead": horizon_step,
            })

    comparison_scored = pd.DataFrame(recursive_rows).merge(
        df[["id", "store_nbr", "item_nbr", "rmsse_scale"]].drop_duplicates("id"),
        on="id", how="left", validate="many_to_one")
    comparison_scored = comparison_scored.merge(
        comparison_reference[["id", "date", "actual"]].rename(columns={"actual":"verified_actual"}),
        on=["id", "date"], how="left", validate="one_to_one")
    if len(comparison_scored) != EXPECTED_COMPARISON_ROWS or comparison_scored.verified_actual.isna().any():
        raise ValueError("Recursive TFT comparison predictions are incomplete")
    if not np.array_equal(comparison_scored.actual.to_numpy(), comparison_scored.verified_actual.to_numpy()):
        raise ValueError("Recursive TFT comparison actuals do not match the verified reference")

    def calculate_metrics(scored):
        actual = scored.actual.to_numpy(dtype=float)
        predicted = scored.prediction.to_numpy(dtype=float)
        errors = actual - predicted
        denominator = float(np.abs(actual).sum())
        per_series = []
        for _, group in scored.groupby("id", sort=False):
            scale = float(group.rmsse_scale.iloc[0])
            if not np.isfinite(scale) or scale <= 0:
                raise ValueError(f"Invalid training-only RMSSE scale for {group.id.iloc[0]}")
            per_series.append(float(np.sqrt(np.mean((group.actual.to_numpy(dtype=float) -
                                                       group.prediction.to_numpy(dtype=float)) ** 2) / scale)))
        return {
            "wape": float(np.abs(errors).sum() / denominator) if denominator else float("nan"),
            "rmsse_mean": float(np.mean(per_series)),
            "rmse": float(np.sqrt(np.mean(errors ** 2))),
            "series_count": int(scored.id.nunique()),
            "observations": int(len(scored)),
        }

    primary_metrics = calculate_metrics(primary_scored)
    comparison_metrics = calculate_metrics(comparison_scored)
    primary_predictions_path = OUTPUT_DIR / "tft_predictions.csv"
    primary_scored[["date", "id", "store_nbr", "item_nbr", "actual", "prediction"]].to_csv(
        primary_predictions_path, index=False)
    comparison_predictions_path = OUTPUT_DIR / "tft_model_comparison_predictions.csv"
    comparison_scored[["date", "id", "store_nbr", "item_nbr", "actual", "prediction", "forecast_lead"]].to_csv(
        comparison_predictions_path, index=False)
    METRICS_PATH = OUTPUT_DIR / "tft_metrics.json"
    metrics = {
        "run_id": RUN_ID,
        "model": "Temporal Fusion Transformer",
        "target": "unit_sales",
        "loss": "QuantileLoss; point forecast is median quantile",
        "gpu": GPU_NAME,
        "architecture": {"hidden_size":8, "attention_head_size":2, "batch_size":BATCH_SIZE,
                      "train_batch_fraction_per_epoch":TRAIN_BATCH_FRACTION, "max_epochs":15, "early_stopping_patience":3},
        "training_end": str(TRAIN_END.date()),
        "model_selection_period": {"start":str(SELECTION_START.date()), "end":str(SELECTION_END.date())},
        "primary_validation": {
            "protocol":"rolling one-day-ahead, actual history through previous day",
            "start":str(VALIDATION_START.date()), "end":str(VALIDATION_END.date()),
            "metrics":primary_metrics,
        },
        "model_comparison_holdout": {
            "protocol":"recursive 28-day forecast from July 18; forecast-generated lag features",
            "start":str(VALIDATION_START.date()), "end":str(VALIDATION_END.date()),
            "same_series_date_keys_as_reference":True,
            "training_scope_note":"Global TFT fit uses all 116 prepared series; score is on the exact 6-series recursive benchmark sample.",
            "metrics":comparison_metrics,
        },
        "metric_definitions":"WAPE=sum(abs(actual-prediction))/sum(abs(actual)); RMSE=sqrt(mean squared error); RMSSE=mean across series of sqrt(series MSE / mean squared first difference in unit_sales strictly before 2017-07-19), matching src/evaluation/metrics.py.",
        "predictions_csv":str(primary_predictions_path),
        "model_comparison_predictions_csv":str(comparison_predictions_path),
        "checkpoint":str(CHECKPOINT_PATH),
        "training_history":str(HISTORY_PATH),
    }
    METRICS_PATH.write_text(json.dumps(metrics, indent=2, default=str))
    EVALUATION_SUCCEEDED = True
    REPORT.update({
        "heldout_evaluation_succeeded":True,
        "metrics_file":METRICS_PATH.name,
        "predictions_file":primary_predictions_path.name,
        "model_comparison_predictions_file":comparison_predictions_path.name,
        "primary_validation_metrics":primary_metrics,
        "model_comparison_metrics":comparison_metrics,
        "evaluation_keys_verified":True,
    })
    write_report(status="success", stage="complete", training_succeeded=True,
                 heldout_evaluation_succeeded=True)
    print(json.dumps(metrics, indent=2))
except Exception as exc:
    EVALUATION_SUCCEEDED = False
    for name in ["tft_predictions.csv", "tft_model_comparison_predictions.csv", "tft_metrics.json", "tft_colab_artifacts.zip"]:
        path = OUTPUT_DIR / name
        if path.exists():
            path.unlink()
    write_report(status="failed", stage="heldout_prediction_and_scoring",
                 heldout_evaluation_succeeded=False, error=f"{type(exc).__name__}: {exc}")
    raise

## Validate and download artifacts

The final cell blocks the ZIP unless this run completed GPU training and both held-out evaluations, and each required artifact exists and belongs to the current run.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

if not REPORT_PATH.is_file():
    raise FileNotFoundError("Execution report is missing; no verified TFT run exists.")
report_data = json.loads(REPORT_PATH.read_text())
if (report_data.get("status") != "success"
        or report_data.get("run_id") != RUN_ID
        or not report_data.get("training_succeeded")
        or not report_data.get("heldout_evaluation_succeeded")
        or not TRAINING_SUCCEEDED or not EVALUATION_SUCCEEDED):
    raise RuntimeError("Download blocked: this run did not finish GPU training and both held-out evaluations.")
required_outputs = [
    OUTPUT_DIR / "tft_predictions.csv",
    OUTPUT_DIR / "tft_model_comparison_predictions.csv",
    OUTPUT_DIR / "tft_metrics.json",
    OUTPUT_DIR / "tft_model.ckpt",
    OUTPUT_DIR / "tft_training_history.csv",
    OUTPUT_DIR / "tft_execution_report.json",
]
missing = [str(path) for path in required_outputs if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError(f"Download blocked; required run artifacts are missing or empty: {missing}")
metrics_check = json.loads((OUTPUT_DIR / "tft_metrics.json").read_text())
if metrics_check.get("run_id") != RUN_ID:
    raise RuntimeError("Download blocked: metrics JSON is not from the current notebook run.")
if metrics_check["primary_validation"]["metrics"]["observations"] != EXPECTED_PRIMARY_ROWS:
    raise RuntimeError("Download blocked: primary metrics do not cover the verified 116-series holdout.")
if metrics_check["model_comparison_holdout"]["metrics"]["observations"] != EXPECTED_COMPARISON_ROWS:
    raise RuntimeError("Download blocked: comparison metrics do not cover the verified benchmark holdout.")
ZIP_PATH = OUTPUT_DIR / "tft_colab_artifacts.zip"
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in required_outputs:
        archive.write(path, arcname=path.name)
print("Created:", ZIP_PATH)
files.download(str(ZIP_PATH))